# NHÓM 11 – ĐỀ TÀI 7: PHÂN LOẠI HOA TRÊN BỘ DỮ LIỆU OXFORD FLOWERS 102
**Học phần:** Nhập môn Học máy (FIT4013) – Trường Đại học Đại Nam
**Giảng viên hướng dẫn:** ThS. Lê Thị Thùy Trang
**Thành viên thực hiện:**
1. Đỗ Tiến Sơn
2. Đỗ Huy Anh Duy

---
### Tóm tắt nội dung notebook:
Notebook này thực hiện trọn vẹn quy trình học máy (Machine Learning Pipeline) khép kín:
1. **EDA & Khám phá dữ liệu:** Thống kê phân phối 102 lớp, kiểm tra tính toàn vẹn và trực quan hóa ảnh mẫu.
2. **Trích xuất đặc trưng đa luồng:** So sánh Luồng A (Raw Pixel 32×32) và Luồng B (Histogram of Oriented Gradients - HOG 64×64).
3. **Chống rò rỉ dữ liệu (Data Leakage):** Scaler chỉ fit trên Train; Validation dùng dò tìm siêu tham số; Test độc lập 100%.
4. **Huấn luyện & Tối ưu:** Khảo sát 11 kịch bản trên K-NN và Cây quyết định (Decision Tree).
5. **Đánh giá & Phân tích lỗi:** Đo lường Accuracy, Precision-Macro, Recall-Macro, F1-Macro, Confusion Matrix và trực quan hóa các ca nhầm lẫn.
6. **Demo suy diễn:** Nhận diện hoa với Top-3 ứng viên kèm tên song ngữ Việt - Anh.

## 1. Cài đặt thư viện và chuẩn bị môi trường

In [ ]:
# Cài đặt các thư viện cần thiết (nếu chạy trên Colab)
!pip -q install numpy pandas matplotlib seaborn scikit-learn scikit-image torch torchvision scipy Pillow joblib

import time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from torchvision.datasets import Flowers102

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Cấu hình đường dẫn và tham số ngẫu nhiên
RANDOM_STATE = 42
PIXEL_SIZE = (32, 32)
HOG_SIZE = (64, 64)

ROOT = Path(".").resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True, parents=True)
print(f"Thư mục gốc: {ROOT} | Thư mục data: {DATA_DIR} | Thư mục outputs: {OUT}")

## 2. Nạp dữ liệu và Phân tích khám phá (EDA)

In [ ]:
def load_split(split):
    try:
        ds = Flowers102(root=str(DATA_DIR), split=split, download=False)
    except Exception:
        print(f"Đang tự động tải tập {split}...")
        ds = Flowers102(root=str(DATA_DIR), split=split, download=True)
    images, labels = [], []
    for img, label in ds:
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))
        img = img.convert("RGB")
        images.append(img.copy())
        labels.append(int(label))
    return images, np.asarray(labels, dtype=np.int64)

print("Đang nạp dữ liệu từ Oxford Flowers 102...")
t0 = time.time()
train_imgs, y_train = load_split("train")
val_imgs, y_val = load_split("val")
test_imgs, y_test = load_split("test")
print(f"-> Nạp thành công trong {time.time()-t0:.2f}s.")
print(f"-> Train: {len(y_train)} ảnh | Val: {len(y_val)} ảnh | Test: {len(y_test)} ảnh.")

# Thống kê phân chia tập dữ liệu
split_summary = pd.DataFrame([
    {"Tập": "Train", "Số ảnh": len(y_train), "Số lớp": len(np.unique(y_train)), "Min/lớp": int(pd.Series(y_train).value_counts().min()), "Max/lớp": int(pd.Series(y_train).value_counts().max())},
    {"Tập": "Validation", "Số ảnh": len(y_val), "Số lớp": len(np.unique(y_val)), "Min/lớp": int(pd.Series(y_val).value_counts().min()), "Max/lớp": int(pd.Series(y_val).value_counts().max())},
    {"Tập": "Test", "Số ảnh": len(y_test), "Số lớp": len(np.unique(y_test)), "Min/lớp": int(pd.Series(y_test).value_counts().min()), "Max/lớp": int(pd.Series(y_test).value_counts().max())}
])
split_summary.to_csv(OUT / "dataset_summary.csv", index=False)
display(split_summary)

In [ ]:
# Biểu đồ phân phối số lượng mẫu trên 102 lớp
all_labels = np.concatenate([y_train, y_val, y_test])
counts = pd.Series(all_labels + 1).value_counts().sort_index()
plt.figure(figsize=(16, 4))
counts.plot(kind="bar", width=0.85, color="steelblue")
plt.title("Phân phối số lượng ảnh trên 102 lớp hoa (Oxford Flowers 102)", fontsize=13, fontweight="bold")
plt.xlabel("Mã lớp hoa (1-102)"); plt.ylabel("Số lượng ảnh")
plt.tight_layout(); plt.savefig(OUT / "class_distribution.png", dpi=200); plt.show()

# Trực quan hóa các mẫu ảnh đại diện
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, idx in zip(axes.ravel(), np.linspace(0, len(train_imgs) - 1, 10, dtype=int)):
    ax.imshow(train_imgs[idx])
    ax.set_title(f"Lớp: {y_train[idx] + 1}", fontsize=11, fontweight="bold")
    ax.axis("off")
fig.suptitle("Một số ảnh mẫu đại diện trong tập huấn luyện (Train Set)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT / "sample_images.png", dpi=200); plt.show()

## 3. Tiền xử lý và Trích xuất đặc trưng (Pixel & HOG)

In [ ]:
# Luồng A: Đặc trưng Pixel 32x32 RGB (3.072 chiều)
def pixel_features(images):
    feats = []
    for img in images:
        arr = np.asarray(img.resize(PIXEL_SIZE), dtype=np.float32) / 255.0
        feats.append(arr.reshape(-1))
    return np.asarray(feats, dtype=np.float32)

# Luồng B: Đặc trưng HOG 64x64 Grayscale (1.764 chiều)
def compute_hog_single(img_array, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    H, W = img_array.shape
    gx = np.zeros_like(img_array)
    gy = np.zeros_like(img_array)
    gx[:, 1:-1] = img_array[:, 2:] - img_array[:, :-2]
    gy[1:-1, :] = img_array[2:, :] - img_array[:-2, :]
    
    mag = np.sqrt(gx**2 + gy**2)
    ori = np.arctan2(gy, gx) * (180.0 / np.pi) % 180.0
    
    cx, cy = pixels_per_cell
    n_cells_y = H // cy
    n_cells_x = W // cx
    
    hist = np.zeros((n_cells_y, n_cells_x, orientations), dtype=np.float32)
    bin_width = 180.0 / orientations
    bin_idx = np.clip((ori / bin_width).astype(int), 0, orientations - 1)
    
    for b in range(orientations):
        mask = (bin_idx == b)
        w_mag = mag * mask
        hist[:, :, b] = w_mag.reshape(n_cells_y, cy, n_cells_x, cx).sum(axis=(1, 3))
                
    bx, by = cells_per_block
    n_blocks_y = n_cells_y - by + 1
    n_blocks_x = n_cells_x - bx + 1
    
    blocks = []
    eps = 1e-5
    for i in range(n_blocks_y):
        for j in range(n_blocks_x):
            block = hist[i:i+by, j:j+bx, :].ravel()
            norm = np.sqrt(np.sum(block**2) + eps)
            block = block / norm
            block = np.clip(block, 0, 0.2)
            norm = np.sqrt(np.sum(block**2) + eps)
            block = block / norm
            blocks.append(block)
            
    return np.concatenate(blocks), mag

def hog_features(images):
    feats = []
    for img in images:
        gray = np.asarray(img.convert("L").resize(HOG_SIZE), dtype=np.float32) / 255.0
        f, _ = compute_hog_single(gray)
        feats.append(f)
    return np.asarray(feats, dtype=np.float32)

print("Đang trích xuất đặc trưng...")
t0 = time.time()
Xtr_pix, Xv_pix, Xte_pix = pixel_features(train_imgs), pixel_features(val_imgs), pixel_features(test_imgs)
Xtr_hog, Xv_hog, Xte_hog = hog_features(train_imgs), hog_features(val_imgs), hog_features(test_imgs)
print(f"-> Trích xuất xong trong {time.time()-t0:.2f}s.")
print(f"-> Kích thước Pixel: Train {Xtr_pix.shape}, Val {Xv_pix.shape}, Test {Xte_pix.shape}")
print(f"-> Kích thước HOG: Train {Xtr_hog.shape}, Val {Xv_hog.shape}, Test {Xte_hog.shape}")

In [ ]:
# Trực quan hóa đối chiếu ảnh gốc và bản đồ Gradient HOG
demo_img = train_imgs[0]
demo_gray = np.asarray(demo_img.convert("L").resize(HOG_SIZE), dtype=np.float32) / 255.0
_, demo_mag = compute_hog_single(demo_gray)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
ax1.imshow(demo_img); ax1.set_title("Ảnh gốc (Input RGB)", fontweight="bold"); ax1.axis("off")
im2 = ax2.imshow(demo_mag, cmap="inferno"); ax2.set_title("Bản đồ Gradient HOG (64x64)", fontweight="bold"); ax2.axis("off")
fig.colorbar(im2, ax=ax2, orientation="vertical", shrink=0.8)
plt.tight_layout(); plt.show()

## 4. Chuẩn hóa đặc trưng & Chống rò rỉ dữ liệu (Data Leakage)

In [ ]:
class Scaler:
    def __init__(self):
        self.mean = None
        self.scale = None

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.scale = np.std(X, axis=0)
        self.scale[self.scale == 0] = 1.0
        return self

    def transform(self, X):
        return (X - self.mean) / self.scale

    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Nguyên tắc vàng: Scaler CHỈ ĐƯỢC FIT trên Train!
scaler_pix = Scaler().fit(Xtr_pix)
Xtr_pix_sc = scaler_pix.transform(Xtr_pix)
Xv_pix_sc = scaler_pix.transform(Xv_pix)

scaler_hog = Scaler().fit(Xtr_hog)
Xtr_hog_sc = scaler_hog.transform(Xtr_hog)
Xv_hog_sc = scaler_hog.transform(Xv_hog)
print("Đã hoàn thành chuẩn hóa đặc trưng độc lập chống rò rỉ dữ liệu!")

## 5. Cài đặt mô hình K-NN và Decision Tree

In [ ]:
class TorchKNN:
    def __init__(self, n_neighbors=5, weights="distance"):
        self.k = n_neighbors
        self.weights = weights
        self.X_train = None
        self.y_train = None
        self.n_classes_ = 102

    def fit(self, X, y):
        self.X_train = torch.tensor(X, dtype=torch.float32)
        self.y_train = torch.tensor(y, dtype=torch.long)
        self.n_classes_ = int(np.max(y) + 1)
        return self

    def predict_proba(self, X):
        X_test = torch.tensor(X, dtype=torch.float32)
        probas = []
        chunk_size = 500
        for i in range(0, len(X_test), chunk_size):
            chunk = X_test[i:i+chunk_size]
            dists = torch.cdist(chunk, self.X_train)
            topk_dists, topk_indices = torch.topk(dists, self.k, dim=1, largest=False)
            topk_labels = self.y_train[topk_indices]
            
            if self.weights == "distance":
                weights = 1.0 / (topk_dists + 1e-5)
            else:
                weights = torch.ones_like(topk_dists)

            chunk_probs = torch.zeros((len(chunk), self.n_classes_), dtype=torch.float32)
            for j in range(len(chunk)):
                chunk_probs[j].scatter_add_(0, topk_labels[j], weights[j])
                s = chunk_probs[j].sum()
                if s > 0:
                    chunk_probs[j] /= s
            probas.append(chunk_probs.numpy())
        return np.vstack(probas)

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, probas=None):
        self.feature = feature; self.threshold = threshold
        self.left = left; self.right = right
        self.value = value; self.probas = probas

    @property
    def is_leaf(self):
        return self.value is not None

class CARTDecisionTree:
    def __init__(self, max_depth=10, min_samples_split=6, n_classes=102, random_state=42):
        self.max_depth = max_depth; self.min_samples_split = min_samples_split
        self.n_classes = n_classes; self.random_state = random_state; self.root = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.root = self._grow_tree(X, y, depth=0)
        return self

    def _gini(self, y):
        counts = np.bincount(y, minlength=self.n_classes)
        probs = counts / len(y)
        return 1.0 - np.sum(probs ** 2)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        counts = np.bincount(y, minlength=self.n_classes)
        leaf_val = int(np.argmax(counts))
        leaf_prob = counts / n_samples

        if (self.max_depth is not None and depth >= self.max_depth) or len(np.unique(y)) == 1 or n_samples < self.min_samples_split:
            return Node(value=leaf_val, probas=leaf_prob)

        n_candidates = min(120, n_features)
        feat_idxs = np.random.choice(n_features, size=n_candidates, replace=False)
        best_feat, best_thresh = None, None
        best_gain = -1.0
        current_gini = self._gini(y)

        for feat in feat_idxs:
            col = X[:, feat]
            thresholds = np.percentile(col, [20, 35, 50, 65, 80])
            for thresh in thresholds:
                left_mask = col <= thresh
                right_mask = ~left_mask
                nl, nr = np.sum(left_mask), np.sum(right_mask)
                if nl < 2 or nr < 2: continue
                gini_split = (nl / n_samples) * self._gini(y[left_mask]) + (nr / n_samples) * self._gini(y[right_mask])
                gain = current_gini - gini_split
                if gain > best_gain:
                    best_gain = gain; best_feat = feat; best_thresh = thresh

        if best_gain <= 0 or best_feat is None:
            return Node(value=leaf_val, probas=leaf_prob)

        left_mask = X[:, best_feat] <= best_thresh
        left_child = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._grow_tree(X[~left_mask], y[~left_mask], depth + 1)
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def _predict_row(self, node, x):
        if node.is_leaf: return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])

def calc_metrics(y_true, y_pred, n_classes=102):
    acc = float(np.mean(y_true == y_pred))
    cm = np.bincount(n_classes * y_true + y_pred, minlength=n_classes * n_classes).reshape(n_classes, n_classes)
    precisions, recalls, f1s = [], [], []
    for c in range(n_classes):
        tp = cm[c, c]
        fp = np.sum(cm[:, c]) - tp
        fn = np.sum(cm[c, :]) - tp
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        precisions.append(p); recalls.append(r); f1s.append(f)
    return {
        "accuracy": acc,
        "precision_macro": float(np.mean(precisions)),
        "recall_macro": float(np.mean(recalls)),
        "f1_macro": float(np.mean(f1s)),
        "confusion_matrix": cm
    }

## 6. Dò tìm siêu tham số trên Validation (11 Kịch bản thử nghiệm)

In [ ]:
candidates = []
print("--- BẮT ĐẦU DÒ TÌM SIÊU THAM SỐ TRÊN TẬP VALIDATION ---")

# 1. KNN Pixel
for k in [1, 3, 5, 7]:
    knn = TorchKNN(n_neighbors=k).fit(Xtr_pix_sc, y_train)
    m = calc_metrics(y_val, knn.predict(Xv_pix_sc))
    candidates.append(("KNN_Pixel", k, m["f1_macro"], m["accuracy"], "pixel"))
    print(f"KNN Pixel (k={k}): Val Acc = {m['accuracy']*100:.2f}%, Val Macro-F1 = {m['f1_macro']:.4f}")

# 2. KNN HOG
for k in [1, 3, 5, 7]:
    knn = TorchKNN(n_neighbors=k).fit(Xtr_hog_sc, y_train)
    m = calc_metrics(y_val, knn.predict(Xv_hog_sc))
    candidates.append(("KNN_HOG", k, m["f1_macro"], m["accuracy"], "hog"))
    print(f"KNN HOG (k={k}): Val Acc = {m['accuracy']*100:.2f}%, Val Macro-F1 = {m['f1_macro']:.4f}")

# 3. Decision Tree HOG
for depth in [10, 20, None]:
    dt = CARTDecisionTree(max_depth=depth, random_state=RANDOM_STATE).fit(Xtr_hog_sc, y_train)
    m = calc_metrics(y_val, dt.predict(Xv_hog_sc))
    candidates.append(("DecisionTree_HOG", str(depth), m["f1_macro"], m["accuracy"], "hog"))
    print(f"Decision Tree HOG (depth={depth}): Val Acc = {m['accuracy']*100:.2f}%, Val Macro-F1 = {m['f1_macro']:.4f}")

val_df = pd.DataFrame([
    {"Thử nghiệm": c[0], "Tham số": str(c[1]), "Val Accuracy (%)": round(c[3]*100, 2), "Val Macro-F1": round(c[2], 4)}
    for c in candidates
]).sort_values("Val Macro-F1", ascending=False)
val_df.to_csv(OUT / "validation_model_selection.csv", index=False)
display(val_df)

## 7. Huấn luyện lại trên (Train + Val) & Đánh giá trên tập Test (6.149 ảnh)

In [ ]:
# Chọn cấu hình tốt nhất của từng họ
selected = {}
for fam in ["KNN_Pixel", "KNN_HOG", "DecisionTree_HOG"]:
    fam_list = [c for c in candidates if c[0] == fam]
    selected[fam] = max(fam_list, key=lambda x: x[2])

print("--- CẤU HÌNH TỐI ƯU ĐƯỢC CHỌN ---")
for fam, s in selected.items():
    print(f"* {fam}: tham số = {s[1]} (Val Macro-F1 = {s[2]:.4f})")

# Ghép dữ liệu Train + Val
y_fit = np.concatenate([y_train, y_val])
X_fit_pix = np.concatenate([Xtr_pix, Xv_pix])
X_fit_hog = np.concatenate([Xtr_hog, Xv_hog])

sc_pix_final = Scaler().fit(X_fit_pix)
X_fit_pix_sc, X_te_pix_sc = sc_pix_final.transform(X_fit_pix), sc_pix_final.transform(Xte_pix)

sc_hog_final = Scaler().fit(X_fit_hog)
X_fit_hog_sc, X_te_hog_sc = sc_hog_final.transform(X_fit_hog), sc_hog_final.transform(Xte_hog)

test_rows = []
cm_dict = {}
err_dict = {}
saved_models = {}

for fam, (name, param, val_f1, _, feat_type) in selected.items():
    t0 = time.time()
    if fam == "KNN_Pixel":
        model = TorchKNN(n_neighbors=int(param)).fit(X_fit_pix_sc, y_fit)
        fit_t = time.time() - t0
        t_p = time.time(); y_pred = model.predict(X_te_pix_sc); pred_t = time.time() - t_p
    elif fam == "KNN_HOG":
        model = TorchKNN(n_neighbors=int(param)).fit(X_fit_hog_sc, y_fit)
        fit_t = time.time() - t0
        t_p = time.time(); y_pred = model.predict(X_te_hog_sc); pred_t = time.time() - t_p
        saved_models["KNN_HOG"] = (model, sc_hog_final)
    else:
        d = None if param == "None" else int(param)
        model = CARTDecisionTree(max_depth=d, random_state=RANDOM_STATE).fit(X_fit_hog_sc, y_fit)
        fit_t = time.time() - t0
        t_p = time.time(); y_pred = model.predict(X_te_hog_sc); pred_t = time.time() - t_p
    
    m = calc_metrics(y_test, y_pred)
    cm_dict[fam] = m["confusion_matrix"]
    wrong = np.where(y_pred != y_test)[0]
    err_dict[fam] = pd.DataFrame({"test_idx": wrong, "true": y_test[wrong] + 1, "pred": y_pred[wrong] + 1})
    
    test_rows.append({
        "Mô hình": fam, "Đặc trưng": feat_type, "Tham số": str(param),
        "Accuracy (%)": round(m["accuracy"]*100, 2),
        "Precision-Macro (%)": round(m["precision_macro"]*100, 2),
        "Recall-Macro (%)": round(m["recall_macro"]*100, 2),
        "Macro-F1": round(m["f1_macro"], 4),
        "Thời gian Fit (s)": round(fit_t, 2),
        "Thời gian Predict (s)": round(pred_t, 2)
    })

metrics_df = pd.DataFrame(test_rows).sort_values("Macro-F1", ascending=False)
display(metrics_df)

## 8. Trực quan hóa so sánh mô hình & Ma trận nhầm lẫn

In [ ]:
# Biểu đồ so sánh các chỉ số
plt.figure(figsize=(10, 5))
plot_df = metrics_df.set_index("Mô hình")[["Accuracy (%)", "Precision-Macro (%)", "Recall-Macro (%)"]]
plot_df.plot(kind="bar", figsize=(10, 5), width=0.7, edgecolor="black")
plt.title("So sánh hiệu năng giữa các mô hình trên tập Test (6.149 ảnh)", fontsize=12, fontweight="bold")
plt.ylabel("Tỷ lệ (%)"); plt.xticks(rotation=0); plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout(); plt.savefig(OUT / "model_comparison.png", dpi=200); plt.show()

# Ma trận nhầm lẫn của 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, fam in zip(axes, ["KNN_Pixel", "KNN_HOG", "DecisionTree_HOG"]):
    sns.heatmap(cm_dict[fam], cmap="Blues", cbar=True, ax=ax)
    ax.set_title(f"Confusion Matrix - {fam}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

## 9. Phân tích lỗi (Error Analysis) & Trực quan hóa các ca nhầm lẫn

In [ ]:
# Top các cặp nhầm lẫn nhiều nhất của KNN HOG
top_pairs = err_dict["KNN_HOG"].value_counts(subset=["true", "pred"]).reset_index(name="count").head(10)
print("--- TOP 10 CẶP LOÀI HOA BỊ NHẦM LẪN NHIỀU NHẤT (KNN HOG) ---")
display(top_pairs)

# Hiển thị minh họa 1 ca nhầm lẫn tiêu biểu
first_wrong = err_dict["KNN_HOG"].iloc[0]
wrong_test_idx = int(first_wrong["test_idx"])
true_cls = int(first_wrong["true"])
pred_cls = int(first_wrong["pred"])

plt.figure(figsize=(5, 4))
plt.imshow(test_imgs[wrong_test_idx])
plt.title(f"Ảnh Test #{wrong_test_idx}\nThực tế: Lớp {true_cls} | Bị đoán nhầm: Lớp {pred_cls}", color="darkred", fontweight="bold")
plt.axis("off"); plt.show()

## 10. Demo suy diễn phân loại hoa với từ điển song ngữ Việt - Anh

In [ ]:
# Nạp module gui_app để thực hiện demo dự đoán nhanh
from src.gui_app import FlowerPredictor, CATEGORY_NAMES

# Lưu mô hình tốt nhất nếu chưa có
best_knn, best_sc = saved_models["KNN_HOG"]
torch.save({
    "X_train": best_knn.X_train, "y_train": best_knn.y_train,
    "k": best_knn.k, "weights": best_knn.weights,
    "scaler_mean": best_sc.mean, "scaler_scale": best_sc.scale
}, OUT / "best_knn_hog_model.pt")

predictor = FlowerPredictor(OUT / "best_knn_hog_model.pt")

# Thử nghiệm trên ảnh hoa hướng dương tiêu biểu
demo_path = DATA_DIR / "flowers-102" / "jpg" / "image_05400.jpg"
if demo_path.exists():
    img_demo = Image.open(demo_path)
    results, mag, elapsed = predictor.predict_image(img_demo, top_n=3)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_demo); ax1.axis("off"); ax1.set_title("Ảnh đầu vào (Input)", fontweight="bold")
    ax2.imshow(mag, cmap="inferno"); ax2.axis("off"); ax2.set_title("Bản đồ HOG Gradient", fontweight="bold")
    plt.show()
    
    print(f"=== KẾT QUẢ DỰ ĐOÁN (Thời gian: {elapsed:.2f} ms) ===")
    for i, r in enumerate(results):
        print(f"Top {i+1}: {r['vn_name']} ({r['en_name']}) - Lớp {r['class_id']} | Độ tin cậy: {r['prob']*100:.2f}%")